In [1]:
!pip install -q transformers accelerate datasets scipy
import torch
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  {i}: {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")

GPUs: 2
  0: Tesla T4  15.6GB
  1: Tesla T4  15.6GB


In [2]:
import torch, time
import numpy as np
from scipy.stats import wilcoxon
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16,
    device_map="auto",              # shard across both T4s
    attn_implementation="eager",    # needed to read attention weights
)
model.eval()

cfg = model.config
NUM_HEADS = cfg.num_attention_heads
dims = {"num_kv_heads": cfg.num_key_value_heads,
        "head_dim": cfg.hidden_size // cfg.num_attention_heads}

print(f"layers={cfg.num_hidden_layers}  heads={NUM_HEADS}")
print("device map sample:", {k: v for k, v in list(model.hf_device_map.items())[:3]},
      "...", {k: v for k, v in list(model.hf_device_map.items())[-2:]})
for i in range(2):
    print(f"GPU{i} allocated: {torch.cuda.memory_allocated(i)/1e9:.1f}GB")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

layers=32  heads=32
device map sample: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0} ... {'model.rotary_emb': 1, 'lm_head': 1}
GPU0 allocated: 7.2GB
GPU1 allocated: 7.2GB


In [3]:
TARGET_LAYER = 31
BOUNDARY_L = 12
SEED = 0
N_SAMPLES = 150
INPUT_DEV = 0   # embed_tokens is on GPU 0

captured, hook_handles = {}, []
def make_k_hook(li):
    def hook(m, i, o):
        b, s, _ = o.shape
        captured[li] = o.view(b, s, dims["num_kv_heads"], dims["head_dim"]) \
                        .transpose(1, 2).detach().cpu()   # -> CPU immediately (sharding-safe)
    return hook
for i, layer in enumerate(model.model.layers):
    if i in (BOUNDARY_L, TARGET_LAYER):     # only the two layers we need
        hook_handles.append(layer.self_attn.k_proj.register_forward_hook(make_k_hook(i)))

ds = load_dataset("dgslibisey/MuSiQue", split="validation")
print(f"hooks on layers {BOUNDARY_L},{TARGET_LAYER}; {len(ds)} examples")

def build_sample(example, n_distractors=4, seed=SEED):
    rng = np.random.default_rng(seed)
    paras = example["paragraphs"]
    sup = [p for p in paras if p["is_supporting"]]
    dis = [p for p in paras if not p["is_supporting"]]
    chosen = list(rng.choice(dis, size=min(n_distractors, len(dis)), replace=False))
    chunks = sup + chosen; rng.shuffle(chunks)
    chunk_ids = [tokenizer(p["paragraph_text"], return_tensors="pt",
                           add_special_tokens=False)["input_ids"][0] for p in chunks]
    q_ids = tokenizer(example["question"] + " Answer within 5 words.",
                      return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    bos = torch.tensor([tokenizer.bos_token_id])
    full_ids = torch.cat([bos] + chunk_ids + [q_ids]).unsqueeze(0)
    chunk_ranges, pos = [], 1
    for c in chunk_ids:
        chunk_ranges.append((pos, pos + len(c))); pos += len(c)
    return {"full_ids": full_ids, "chunk_ids": chunk_ids,
            "chunk_ranges": chunk_ranges, "query_range": (pos, pos + len(q_ids))}

def compute_true_dev(sample):
    bos = sample["full_ids"][0, :1]
    chunks_only = torch.cat([bos] + sample["chunk_ids"]).unsqueeze(0).to(INPUT_DEV)
    captured.clear()
    with torch.no_grad(): model(chunks_only)
    K_together = captured[TARGET_LAYER].clone()          # already on CPU
    dev = np.full(chunks_only.shape[1], np.nan)
    for ci, c in enumerate(sample["chunk_ids"]):
        captured.clear()
        with torch.no_grad(): model(torch.cat([bos, c]).unsqueeze(0).to(INPUT_DEV))
        K_alone = captured[TARGET_LAYER][:, :, 1:, :]
        s_, e_ = sample["chunk_ranges"][ci]
        diff = K_together[0, :, s_:e_, :].float() - K_alone[0].float()
        dev[s_:e_] = diff.pow(2).sum((0, 2)).sqrt().numpy()
    return dev

def recall_at_k(pred, true, k_frac=0.20):
    m = ~np.isnan(pred) & ~np.isnan(true)
    p, t = pred[m], true[m]
    if len(p) < 5: return np.nan
    k = max(1, int(k_frac * len(t)))
    return len(set(np.argsort(t)[-k:]) & set(np.argsort(p)[-k:])) / k

N_SINK = 4   # MagicPIG convention: first 4 tokens treated as sink

def sparse_q_variants(sample, chunk_end):
    """One forward pass -> (unmasked, sink-masked) sparse_q mean-head scores,
       plus tail-coverage diagnostics."""
    full_ids = sample["full_ids"].to(INPUT_DEV)
    qs, qe = sample["query_range"]
    with torch.no_grad():
        out = model(full_ids, output_attentions=True)
    A = out.attentions[BOUNDARY_L][0].float().cpu()      # [heads, seq, seq]
    Q = A[:, qs:qe, :]                                    # query rows
    del out

    # --- unmasked (baseline, exactly as before) ---
    row = Q.sum(dim=1)                                    # [heads, seq]
    unmasked = row.mean(dim=0).numpy()[:chunk_end]

    # --- sink-masked: zero attention to first N_SINK positions, renormalize each row ---
    Qm = Q.clone()
    Qm[:, :, :N_SINK] = 0.0
    Qm = Qm / Qm.sum(dim=-1, keepdim=True).clamp(min=1e-9)
    masked = Qm.sum(dim=1).mean(dim=0).numpy()[:chunk_end]

    # --- diagnostics ---
    sink_mass = Q[:, :, :N_SINK].sum(-1).mean().item()    # avg fraction of a query row's mass on sink
    doc = row.mean(dim=0).numpy()[N_SINK:chunk_end]       # doc-token scores, sink excluded
    doc_sorted = np.sort(doc)[::-1]
    k20 = max(1, int(0.20 * len(doc_sorted)))
    tail_cov = doc_sorted[:k20].sum() / max(doc_sorted.sum(), 1e-9)  # mass captured by top-20% doc tokens

    return unmasked, masked, sink_mass, tail_cov

# gate on 1 sample
s0 = build_sample(ds[0])
td0 = compute_true_dev(s0)
ce0 = s0["chunk_ranges"][-1][1]
um, mk, sm, tc = sparse_q_variants(s0, ce0)
print(f"true_dev non-NaN={(~np.isnan(td0)).sum()}, chunk_end={ce0}")
print(f"recall@20%: unmasked={recall_at_k(um, td0):.3f}  sink-masked={recall_at_k(mk, td0):.3f}")
print(f"sink mass fraction={sm:.3f}   top-20% doc-token coverage={tc:.3f}")

musique_ans_v1.0_train.jsonl:   0%|          | 0.00/241M [00:00<?, ?B/s]

musique_ans_v1.0_dev.jsonl:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19938 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2417 [00:00<?, ? examples/s]

hooks on layers 12,31; 2417 examples
true_dev non-NaN=464, chunk_end=465
recall@20%: unmasked=0.435  sink-masked=0.413
sink mass fraction=0.587   top-20% doc-token coverage=0.765


In [4]:
t0 = time.time()
all_e8 = []
for i in range(N_SAMPLES):
    s = build_sample(ds[i])
    td = compute_true_dev(s)
    ce = s["chunk_ranges"][-1][1]
    um, mk, sm, tc = sparse_q_variants(s, ce)
    all_e8.append({"true_dev": td, "unmasked": um, "masked": mk,
                   "sink_mass": sm, "tail_cov": tc})
    if (i + 1) % 25 == 0:
        print(f"{i+1}/{N_SAMPLES}  elapsed={time.time()-t0:.0f}s")
    torch.cuda.empty_cache()
print(f"Done in {time.time()-t0:.0f}s")

25/150  elapsed=53s
50/150  elapsed=102s
75/150  elapsed=163s
100/150  elapsed=223s
125/150  elapsed=283s
150/150  elapsed=337s
Done in 338s


In [5]:
um_r = np.array([recall_at_k(r["unmasked"], r["true_dev"]) for r in all_e8])
mk_r = np.array([recall_at_k(r["masked"],  r["true_dev"]) for r in all_e8])
m = ~np.isnan(um_r) & ~np.isnan(mk_r)
stat, p = wilcoxon(um_r[m], mk_r[m])

print(f"=== E8: sink-mask test (n={m.sum()}, sparse_q mean@12) ===")
print(f"  unmasked   recall@20% = {um_r[m].mean():.3f}")
print(f"  sink-masked recall@20% = {mk_r[m].mean():.3f}")
print(f"  diff = {mk_r[m].mean()-um_r[m].mean():+.4f}   p = {p:.5f}")

sink = np.array([r["sink_mass"] for r in all_e8])
tail = np.array([r["tail_cov"] for r in all_e8])
print(f"\n=== MagicPIG-connection diagnostics ===")
print(f"  sink mass fraction:   mean={sink.mean():.3f}  std={sink.std():.3f}")
print(f"  top-20% doc coverage: mean={tail.mean():.3f}  std={tail.std():.3f}")
print(f"  (MagicPIG reports top-20% covering ~70-80% in their setting)")

# does per-sample tail coverage predict per-sample recall? (peakier attention -> better selection?)
from scipy.stats import spearmanr
rho, p2 = spearmanr(tail[m], um_r[m])
print(f"\n  corr(tail coverage, recall): spearman rho={rho:.3f}  p={p2:.5f}")

=== E8: sink-mask test (n=150, sparse_q mean@12) ===
  unmasked   recall@20% = 0.426
  sink-masked recall@20% = 0.420
  diff = -0.0055   p = 0.00047

=== MagicPIG-connection diagnostics ===
  sink mass fraction:   mean=0.523  std=0.040
  top-20% doc coverage: mean=0.733  std=0.043
  (MagicPIG reports top-20% covering ~70-80% in their setting)

  corr(tail coverage, recall): spearman rho=0.376  p=0.00000
